<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Heuristic Optimization: Greedy Algorithms and Local Search

**Build a feasible delivery tour quickly, then improve it by changing part of its visit order.**

We carry forward the distinction between a decision, feasibility, and its objective value. We now transfer those ideas to a different system: a vehicle visiting five cities. The clock observations, calibration equality, and correction bound no longer apply.

The new decision is a discrete visit order. We use the distance matrix and nearest-neighbor example from the supplied heuristic-optimization materials. A greedy rule constructs a tour; local search revises a complete tour. We keep this same five-city problem throughout the notebook.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Choose a complete tour

A vehicle must visit cities $0,1,2,3,4$ once each and return to its starting city. A **tour** is that complete closed route. Shorter total distance means less travel.

Let $C_{ij}$ be the fixed travel distance from city $i$ to city $j$. The source gives no physical unit, so we report **distance units**. All city pairs are connected, and the distances are symmetric: $C_{ij}=C_{ji}$. The numbers are travel costs; we do not assume they are straight-line distances on a map.

| From / to | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| $0$ | 0 | 2 | 9 | 10 | 7 |
| $1$ | 2 | 0 | 8 | 5 | 6 |
| $2$ | 9 | 8 | 0 | 4 | 3 |
| $3$ | 10 | 5 | 4 | 0 | 1 |
| $4$ | 7 | 6 | 3 | 1 | 0 |

Write the decision as $x=(v_0,v_1,v_2,v_3,v_4,v_5)$, where $v_k$ is the city visited at position $k$. Feasibility requires that $v_0,\ldots,v_4$ contain every city exactly once and $v_5=v_0$. The objective is

$$
f(x)=\sum_{k=0}^{4} C_{v_k,v_{k+1}}.
$$

The final term includes the return trip. For example,

$$
f(2,4,3,1,0,2)=3+1+5+2+9=20.
$$

The distance matrix is fixed data. The visit order is the physical decision. Here the raw performance output, total distance, is also the scalar objective. A route with a missing city or return trip is rejected before any score comparison.


In [ ]:
import numpy as np

# Fixed travel distances, copied from the supplied example_1.py
DISTANCES = np.array([
    [0, 2, 9, 10, 7],
    [2, 0, 8, 5, 6],
    [9, 8, 0, 4, 3],
    [10, 5, 4, 0, 1],
    [7, 6, 3, 1, 0],
], dtype=int)
CITY_COUNT = len(DISTANCES)


def evaluate_tour(tour):
    tour = tuple(tour)
    valid_labels = all(isinstance(city, (int, np.integer))
                       and 0 <= city < CITY_COUNT for city in tour)
    feasible = (len(tour) == CITY_COUNT + 1 and valid_labels
                and tour[0] == tour[-1]
                and set(tour[:-1]) == set(range(CITY_COUNT)))
    if not feasible:
        return {"decision": tour, "feasible": False,
                "edge_distances": None, "objective": None}
    edges = np.array([DISTANCES[a, b] for a, b in zip(tour, tour[1:])])
    return {"decision": tour, "feasible": True,
            "edge_distances": edges, "objective": int(edges.sum())}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The open route $(2,4,3,1,0)$ has distance $3+1+5+2=11$, but it omits the required return trip. Adding the edge from 0 to 2 gives $11+9=20$. The incomplete route is rejected before any score comparison.

The rows below keep the same visit order. The orange return edge closes the lower route and adds 9 distance units. The edge labels give travel distances; line lengths do not represent those distances.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/complete_tour_return.png" alt="Both rows visit cities 2, 4, 3, 1, and 0 in that order, using edges of distance 3, 1, 5, and 2. The open route has distance 11 and is incomplete. The closed tour adds the return edge from 0 to 2, of distance 9, for a feasible total of 20." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>



In [ ]:
for tour in ((2, 4, 3, 1, 0, 2), (2, 4, 3, 1, 0)):
    result = evaluate_tour(tour)
    if not result["feasible"]:
        print(tour, "rejected: visit every city once and return to the start")
        continue
    print(tour, "edge distances:", result["edge_distances"],
          "total:", result["objective"], "distance units")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Build a tour with a greedy rule

A **heuristic** searches for a useful solution without a general guarantee of finding the global optimum. It still has to respect the problem's constraints. A **greedy construction** commits to the best immediate choice under a stated rule, without revising earlier choices.

The **nearest-neighbor rule** chooses the closest unvisited city. If the current city is $i$ and $U$ is the set of unvisited cities, choose

$$
j^\star\in\underset{j\in U}{\operatorname{argmin}}\ C_{ij}.
$$

Here $\operatorname{argmin}$ returns a city attaining the smallest distance. We break ties by choosing the smaller city label. After every city is visited, return to the starting city.

Compare the four labeled distances from city 2. The orange edge selects city 4 because its distance is smallest. This picture shows one choice; line lengths do not represent the distances.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/greedy_next_city.png" alt="One greedy step from city 2 compares distances 9, 8, 4, and 3 to unvisited cities 0, 1, 3, and 4. The orange arrow selects city 4 at distance 3." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>

Starting at city 2, as in the supplied code, the rule gives:

| Current city | Unvisited cities | Chosen next city | Added distance |
|:---|:---|:---|---:|
| 2 | 0, 1, 3, 4 | 4 | 3 |
| 4 | 0, 1, 3 | 3 | 1 |
| 3 | 0, 1 | 1 | 5 |
| 1 | 0 | 0 | 2 |
| 0 | None | Return to 2 | 9 |

The cheap first edge does not account for the later return edge. The rule constructs a feasible tour because every pair is connected, each new city is unvisited, and the route closes at the end. These facts do not prove that its total distance is smallest.


In [ ]:
def nearest_neighbor(start_city):
    if start_city not in range(CITY_COUNT):
        raise ValueError("The starting city must be one of the city labels.")
    remaining = set(range(CITY_COUNT)) - {start_city}
    tour = [start_city]
    trace = []
    while remaining:
        current = tour[-1]
        next_city = min(remaining,
                        key=lambda city: (DISTANCES[current, city], city))
        trace.append({"current": current, "unvisited": tuple(sorted(remaining)),
                      "chosen": next_city,
                      "distance": int(DISTANCES[current, next_city])})
        tour.append(next_city)
        remaining.remove(next_city)
    tour.append(start_city)
    result = evaluate_tour(tour)
    result["trace"] = trace
    return result


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Run the original starting choice. The result should reproduce the worked total of 20 distance units, including the return edge.


In [ ]:
source_run = nearest_neighbor(2)
for step in source_run["trace"]:
    print(step)
print("Tour:", source_run["decision"])
print("Return distance:", source_run["edge_distances"][-1])
print("Total distance:", source_run["objective"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Change only the construction start

Keep the distance matrix, visit requirements, and objective fixed. Change only the city from which the algorithm begins constructing a tour.

In this problem the physical tour is a cycle: rotating its written starting point does not change its edges or distance. No depot, time window, or required departure city is imposed. The construction start is therefore a **hyperparameter** of this search. It can change the selected tour; it cannot change the distance of a fixed tour.

For example, starting at city 1 gives $(1,0,4,3,2,1)$, with distance $2+7+1+4+8=22$. The tour from city 2 has distance 20, so the first result is already a counterexample to a general optimality claim for this greedy rule.


In [ ]:
greedy_results = [nearest_neighbor(start) for start in range(CITY_COUNT)]
for start, result in enumerate(greedy_results):
    print(f"Start {start}: {result['decision']} | distance {result['objective']}")
best_greedy = min(greedy_results, key=lambda result: result["objective"])
print("Best among these five greedy runs:", best_greedy["objective"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Establish an exact reference for this small instance

The five runs above test five constructed tours. They do not enumerate every feasible tour. To find the actual optimum for this tiny instance, fix city 2 as the written start and enumerate all orders of the other four cities.

There are $4!=24$ orders. Reversed tours have the same distance because the matrix is symmetric, so these include both orientations of 12 distinct undirected cycles. Keeping those duplicates is harmless. Every feasible cycle is represented, so the smallest value in this complete enumeration is a **global optimum for this five-city instance**.

Each tile below is the total distance of one complete tour with city 2 fixed as the written start. The 24 values are sorted. Different tours may share a value, and reverse orientations are both included. The two teal tiles attain the minimum of 20 distance units.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/exact_enumeration_coverage.png" alt="Fixing city 2 leaves 4 times 3 times 2 times 1, or 24 complete tours. Their sorted total distances are 20, 20, 21, 21, 22, 22, 22, 22, 24, 24, 25, 25, 30, 30, 31, 31, 33, 33, 33, 33, 34, 34, 35, and 35. Teal highlights the two tours attaining 20." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>

The enumeration below gives $f^\star=20$. The relative gap of a feasible tour is

$$
\operatorname{gap}(x)=\frac{f(x)-f^\star}{f^\star}\times100\%.
$$

The start-1 greedy tour has gap $(22-20)/20\times100\%=10\%$. The start-3 tour has distance 24 and gap $(24-20)/20\times100\%=20\%$. Taking the nearest available edge at each step does not guarantee a zero gap. For a larger problem we may not know $f^\star$; a best-known feasible value is not automatically the optimum.


In [ ]:
from itertools import permutations

reference_start = 2
other_cities = [city for city in range(CITY_COUNT) if city != reference_start]
enumerated_results = [
    evaluate_tour((reference_start,) + order + (reference_start,))
    for order in permutations(other_cities)
]
exact_result = min(enumerated_results, key=lambda result: result["objective"])
exact_distance = exact_result["objective"]
print("Complete tours evaluated:", len(enumerated_results))
print("One globally optimal tour:", exact_result["decision"])
print("Verified optimal distance:", exact_distance)
for start, result in enumerate(greedy_results):
    gap = 100 * (result["objective"] - exact_distance) / exact_distance
    print(f"Start {start}: gap {gap:.1f}%")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 5 · Define one local change: 2-opt

Greedy construction extends an unfinished route. **Local search** starts with a complete feasible tour and repeatedly considers small changes to it. Its **neighborhood**, $N(x)$, is the set of tours reachable from the current tour $x$ by one allowed change.

A **2-opt move** removes two nonadjacent edges and reconnects the pieces into a single cycle. For our symmetric distances, this is implemented by reversing a segment of the city order.

Compare the two orange city tiles in the rows below. Reversing their order keeps every city in the tour, but changes the two connections at the segment boundaries. The small numbers between city tiles are edge distances.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/two_opt_segment_reversal-v3.png" alt="Before 2-opt, the tour 1, 0, 4, 3, 2, 1 has edge distances 2, 7, 1, 4, 8 and total 22. Reversing cities 3 and 2 gives tour 1, 0, 4, 2, 3, 1 with edge distances 2, 7, 3, 4, 5 and total 21." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>

Starting from the length-22 tour, reverse the segment $(3,2)$:

$$
\begin{aligned}
x&=(1,0,4,3,2,1),& f(x)&=22,\\
x'&=(1,0,4,2,3,1),& f(x')&=21.
\end{aligned}
$$

The move removes edges $(4,3)$ and $(2,1)$, with total distance $1+8=9$, and adds $(4,2)$ and $(3,1)$, with total $3+5=8$. The reversed internal edge has the same distance in either direction. Thus the total decreases by 1.

In the code, positions $i$ through $j$ are reversed, with $1\le i<j\le n-1$ and $n=5$. The first and final city stay fixed. Reversing all four internal positions only reverses the same undirected cycle, so that redundant move is excluded. The five remaining moves preserve feasibility. The indices are positions in the tour, not city labels.


In [ ]:
def two_opt_neighbors(tour):
    tour = tuple(tour)
    city_count = len(tour) - 1
    for i in range(1, city_count - 1):
        for j in range(i + 1, city_count):
            if i == 1 and j == city_count - 1:
                continue  # Same undirected cycle, written in reverse.
            candidate = tour[:i] + tour[i:j + 1][::-1] + tour[j + 1:]
            yield (i, j), candidate


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

List the complete 2-opt neighborhood of the start-1 greedy tour. Check feasibility before comparing each distance with 22.


In [ ]:
initial_tour = greedy_results[1]["decision"]
for move, tour in two_opt_neighbors(initial_tour):
    result = evaluate_tour(tour)
    if result["feasible"]:
        change = result["objective"] - greedy_results[1]["objective"]
        print(f"Reverse positions {move}: {tour} | distance {result['objective']}"
              f" | change {change:+d}")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 6 · Repeat strictly improving moves

Use **best improvement**: evaluate every neighbor and choose the one with the smallest distance. Accept it only if its distance is strictly smaller than the current value. Ties between equally good improving moves follow the order in which the code lists them.

Repeat this calculation from the accepted tour. If a full neighborhood contains no strict improvement, stop. The returned tour $x_{\mathrm{local}}$ satisfies

$$
f(x_{\mathrm{local}})\le f(x')
\quad\text{for every }x'\in N(x_{\mathrm{local}}).
$$

This is local optimality **with respect to the chosen neighborhood**. It is not the differentiable stationary-point condition from the clock problem. The city order is discrete, and the allowed moves define what “nearby” means.

Strict improvement prevents revisiting a tour. Since this problem has finitely many tours, the search terminates. Accepting equal-cost moves without another rule could cause cycling. A run stopped early by a search budget would not establish the neighborhood condition above.


In [ ]:
def local_search(initial_tour, neighborhood=two_opt_neighbors):
    current = evaluate_tour(initial_tour)
    if not current["feasible"]:
        raise ValueError("Local search needs a complete feasible initial tour.")
    history = [current]
    accepted_moves = []
    neighbor_evaluations = 0
    while True:
        best = current
        best_move = None
        for move, tour in neighborhood(current["decision"]):
            candidate = evaluate_tour(tour)
            neighbor_evaluations += 1
            if not candidate["feasible"]:
                continue
            if candidate["objective"] < best["objective"]:
                best, best_move = candidate, move
        if best_move is None:
            return {"result": current, "history": history,
                    "accepted_moves": accepted_moves,
                    "neighbor_evaluations": neighbor_evaluations,
                    "stop_reason": "No strictly improving neighbor"}
        current = best
        history.append(current)
        accepted_moves.append(best_move)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

For the start-1 greedy tour, best improvement gives distances $22\to21\to20$. The final scan checks every neighbor once more before stopping. The exact reference from Section 4, rather than the stopping rule alone, confirms that this particular result is global.

Read each column from the orange current distance to its five neighbor distances. Teal marks an accepted improvement. The last column still checks all five neighbors; its smallest value is 21, so it cannot improve the current 20. Every value is in distance units.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/local_search_complete_scans.png" alt="Scan 1 evaluates neighbors 35, 30, 24, 22, and 21 from current distance 22 and accepts 21. Scan 2 evaluates 31, 33, 20, 25, and 22 from 21 and accepts 20. Scan 3 evaluates 30, 33, 21, 24, and 22 from 20 and stops because no neighbor improves it." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>



In [ ]:
improved_run = local_search(initial_tour)
for step, result in enumerate(improved_run["history"]):
    print(f"Accepted moves {step}: {result['decision']}"
          f" | distance {result['objective']}")
print("Reversed position pairs:", improved_run["accepted_moves"])
print("Neighbor evaluations, including the final scan:",
      improved_run["neighbor_evaluations"])
print("Stop:", improved_run["stop_reason"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 7 · A local optimum depends on the neighborhood

Keep the matrix, objective, and current tour fixed at $x=(2,3,1,0,4,2)$, with distance 21. Change only which moves the search may consider.

First allow swaps of consecutive internal positions: $(1,2)$, $(2,3)$, or $(3,4)$. The initial and final city remain fixed. This deliberately restricted neighborhood produces distances 33, 25, and 31, so the tour is a local optimum under this rule.

Now allow the full 2-opt neighborhood. Reversing positions 1 through 3 gives $(2,0,1,3,4,2)$, with distance 20. The old tour was locally optimal for the smaller neighborhood and was not globally optimal.

Each card below shows a complete tour's total distance, not a city label. Compare the same current distance of 21 under the two move rules. All three adjacent-swap neighbors are worse; the lower panel shows one improving 2-opt neighbor.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/local_optimum_neighborhood-v4.png" alt="With current tour distance 21, the three adjacent-position swaps give distances 33, 25, and 31. Allowing 2-opt adds an improving neighbor with distance 20, so local optimality depends on the allowed moves." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>

Only the neighbor generator changes. The same feasibility checks, score, and improvement rule apply to both runs.


In [ ]:
def adjacent_neighbors(tour):
    tour = tuple(tour)
    for i in range(1, len(tour) - 2):
        candidate = list(tour)
        candidate[i], candidate[i + 1] = candidate[i + 1], candidate[i]
        yield (i, i + 1), tuple(candidate)


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Verify the restricted local optimum and the improvement permitted by 2-opt. This example does not establish that 2-opt always reaches a global optimum; it shows why every local-optimality statement must name its neighborhood.


In [ ]:
comparison_tour = (2, 3, 1, 0, 4, 2)
print("Consecutive-position neighbor distances:",
      [evaluate_tour(tour)["objective"]
       for _, tour in adjacent_neighbors(comparison_tour)])
restricted_run = local_search(comparison_tour, adjacent_neighbors)
wider_run = local_search(comparison_tour, two_opt_neighbors)
for name, run in (("Consecutive positions", restricted_run), ("2-opt", wider_run)):
    print(name, "distance:", run["result"]["objective"],
          "accepted moves:", run["accepted_moves"])


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 8 · Compare results while keeping the problem fixed

Apply the same 2-opt search to every greedy result. Compare the initial distance with the final distance for each construction start. In this instance all five searches reach 20; full enumeration independently verifies that value.

The upper plot shows how the construction start changes greedy quality. The lower plot shows the accepted improvements from start 1. The dashed line is the optimum verified by exhaustive enumeration, not an estimate from the plotted runs.


In [ ]:
import matplotlib.pyplot as plt

BLUE = "#2878b5"
TEAL = "#168578"
ORANGE = "#e78b24"
GRAY = "#707070"

all_improved = [local_search(result["decision"]) for result in greedy_results]


def show_search_comparison():
    figure, axes = plt.subplots(2, 1, figsize=(5.6, 7.0))
    starts = np.arange(CITY_COUNT)
    greedy_distances = [result["objective"] for result in greedy_results]
    final_distances = [run["result"]["objective"] for run in all_improved]
    axes[0].bar(starts - 0.18, greedy_distances, width=0.36,
                color=BLUE, label="Greedy tour")
    axes[0].bar(starts + 0.18, final_distances, width=0.36,
                color=TEAL, label="After 2-opt")
    axes[0].set(xticks=starts, xlabel="Construction start (city label)",
                ylabel="Total distance (distance units)", ylim=(0, 28),
                title="The start changes greedy tour quality")
    axes[0].legend(loc="lower left", fontsize=10)

    history = [result["objective"] for result in improved_run["history"]]
    steps = np.arange(len(history))
    axes[1].plot(steps, history, marker="o", color=ORANGE,
                 linewidth=2, label="Current tour from start 1")
    axes[1].set(xticks=steps, xlabel="Number of accepted 2-opt moves",
                ylabel="Total distance (distance units)", ylim=(19.5, 23),
                title="Two accepted moves reduce 22 to 20")
    for step, distance in zip(steps, history):
        axes[1].annotate(str(distance), (step, distance), xytext=(0, 8),
                         textcoords="offset points", ha="center")
    for axis in axes:
        axis.axhline(exact_distance, color=GRAY, linestyle="--",
                     label="Verified optimum: 20", zorder=0)
        axis.grid(alpha=0.25)
        axis.set_axisbelow(True)
    axes[1].legend(loc="upper right", fontsize=10)
    figure.tight_layout()
    return figure


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Draw the two comparisons from the evaluation records. No map coordinates are invented: every distance still comes from the supplied matrix.


In [ ]:
comparison_figure = show_search_comparison()
plt.show()


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Enumeration checks $(n-1)!$ written tours after fixing the start. For $n=5$ that is 24; for $n=10$ it is $9!=362{,}880$. A nearest-neighbor construction compares $4+3+2+1=10$ outgoing edge distances here. A full 2-opt scan considers five neighboring tours; the start-1 run uses three scans, including the final check.

These count different operations, so they are not timing measurements. Our code recalculates each neighbor's full distance for clarity. Heuristics can avoid enumerating every tour, but neither a small search effort nor successful runs on this instance prove a general quality guarantee.

| Method | What it does here | What supports its conclusion |
|:---|:---|:---|
| Greedy construction | Adds the nearest unvisited city | Produces a feasible tour; does not certify global optimality |
| Local search | Replaces a full tour with a strictly better neighbor | A complete final scan verifies local optimality for that neighborhood |
| Exhaustive enumeration | Evaluates all 24 written tours | Proves the global optimum for this finite instance |

Greedy rules can be exact for problems with additional structure and a proof. Here nearest neighbor is a heuristic. “Greedy” describes the choice rule; “heuristic” describes the absence of a general optimality guarantee for this application.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 9 · Source material and method reference

The supplied `Lecture_10_1_Heuristic_Optimization/10_1.pptx` introduces heuristic construction rules. Its `ExampleCode_10_1/example_1.py` supplies the five-city distance matrix, nearest-neighbor rule, and original start at city 2. This notebook develops that one example throughout; it adds exhaustive verification, 2-opt local search, and the restricted-neighborhood comparison.

Michael A. Trick's [Heuristics](https://mat.tepper.cmu.edu/orclass/integer/node17.html) describes nearest-neighbor tour construction and two-opt tour improvement. The worked numbers and comparisons above use the supplied distance matrix.
